In [0]:
%pip install --quiet gspread google-auth
dbutils.library.restartPython()

In [0]:
import os, sys, json, importlib

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(os.path.join(repo_root, "src"))
import sheets_export
importlib.reload(sheets_export)

sa_info = json.loads(dbutils.secrets.get("power-prices", "gcp-sa-json"))
summary = sheets_export.export(spark, sa_info)

print(f"{'spreadsheet':<26} {'rows':>9} {'cols':>5} {'cells':>11}   Excel export (what Tableau downloads)")
for name, rows, cols, cells, status in summary:
    print(f"pp_{name:<23} {rows:>9,} {cols:>5} {cells:>11,}   {status}")

refused = [s[0] for s in summary if s[4].startswith("REFUSED")]
if refused:
    raise RuntimeError(f"Drive refused the Excel export for: {', '.join(refused)}. "
                       "Tableau cannot read these files; they need to be made smaller.")